# 05 — New Models: Mistral, Phi-2, and Gemma

This notebook covers loading and generating with the three new model families added to MyLLM:

| Model | Family | Params | Tokenizer | Auth |
|-------|--------|--------|-----------|------|
| `mistral-7b-v0.1` | Mistral | 7B | SentencePiece (auto-downloaded) | None |
| `phi-2` | Phi | 2.7B | CodeGen-style (bring your own) | None |
| `gemma-2b` | Gemma | 2B | SentencePiece (auto-downloaded) | `HF_TOKEN` |
| `gemma-7b` | Gemma | 7B | SentencePiece (auto-downloaded) | `HF_TOKEN` |

All use the same `LLM.from_pretrained()` API as GPT-2.

> **Colab users:** Run the setup cell, restart the runtime, then continue.

In [ ]:
import subprocess, sys, os

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _is_colab():
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-cache-dir',
         'git+https://github.com/silvaxxx1/MyLLM.git'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Done! Now go to Runtime → Restart runtime, then continue.')
else:
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    has_uv = subprocess.run(['which', 'uv'], capture_output=True).returncode == 0
    cmd = ['uv', 'pip', 'install', '-e', root] if has_uv else [sys.executable, '-m', 'pip', 'install', '-e', root]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Local editable install ready.')

## 1. Inspect ModelConfigs

All four new models are registered in the config registry.
Key things to notice compared to GPT-2:
- `n_query_groups < n_head` → GQA (Mistral) or MQA (Gemma 2B)
- `mlp_hidden_size` is set explicitly (not `4 × n_embd`)
- `scale_embeddings=True` on Gemma

In [ ]:
import torch
from myllm import ModelConfig

models = ['mistral-7b-v0.1', 'phi-2', 'gemma-2b', 'gemma-7b']

for name in models:
    cfg = ModelConfig.from_name(name)
    mem = cfg.estimate_memory(batch_size=1, dtype=torch.float16)
    print(f"{name}")
    print(f"  layers={cfg.n_layer}  heads={cfg.n_head}  kv_heads={cfg.n_query_groups}  embd={cfg.n_embd}  head_size={cfg.head_size}")
    print(f"  mlp_hidden={cfg.mlp_hidden_size}  vocab={cfg.vocab_size}  rope={cfg.use_rope}  scale_emb={cfg.scale_embeddings}")
    print(f"  ~{mem['n_parameters']/1e9:.1f}B params  ({mem['parameters_gb']:.1f} GB at fp16)")
    print()

## 2. Mistral 7B v0.1

Mistral is a drop-in LLaMA-architecture model with two differences:
- **GQA** — 8 KV heads instead of 32 (4× memory saving on attention cache)
- **Sliding window attention** — not yet implemented in MyLLM (full attention used instead)

The tokenizer is the same SentencePiece model as LLaMA 2 — auto-downloaded alongside the weights.
No HuggingFace token required.

In [ ]:
from myllm import LLM, GenerationConfig

# Weights are downloaded in two shards and merged automatically.
# tokenizer.model is downloaded alongside the weights.
llm_mistral = LLM.from_pretrained(
    'mistral-7b-v0.1',
    torch_dtype=torch.float16,   # fp16 halves memory: ~7 GB GPU RAM
)
print(llm_mistral)

In [ ]:
gen_cfg = GenerationConfig(
    max_length=100,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    do_sample=True,
)

result = llm_mistral.generate_text(
    'The key to building reliable AI systems is',
    generation_config=gen_cfg,
    skip_prompt=True,
)
print(result['text'])

## 3. Phi-2

Phi-2 is Microsoft's 2.7B model — unusually capable for its size.
Architecture is GPT-style (LayerNorm, GptMLP) but with partial RoPE (40% of head dims)
and biases throughout.

**Tokenizer note:** Phi-2 uses a custom CodeGen-style tokenizer with a 51200-token
vocabulary that isn't compatible with our SentencePiece or tiktoken implementations.
Weights load automatically; you need to supply a tokenizer manually.
The easiest option on Colab is HuggingFace `transformers`.

In [ ]:
# Phi-2 weights load fine — tokenizer needs to be supplied separately
llm_phi = LLM.from_pretrained(
    'phi-2',
    torch_dtype=torch.float16,
)
# Note printed: "phi-2 tokenizer is not auto-loadable"
print(llm_phi)

In [ ]:
# Option A: use HuggingFace tokenizer as a drop-in
# pip install transformers
from transformers import AutoTokenizer as HFTokenizer

hf_tok = HFTokenizer.from_pretrained('microsoft/phi-2', trust_remote_code=True)

gen_cfg = GenerationConfig(
    max_length=80,
    temperature=0.7,
    top_k=50,
    do_sample=True,
)

# Pass the HF tokenizer directly — generate_text() accepts any tokenizer
# that implements encode(text, return_tensors='pt') and decode(ids)
result = llm_phi.generate_text(
    'def fibonacci(n):',
    tokenizer=hf_tok,
    generation_config=gen_cfg,
    skip_prompt=True,
)
print(result['text'])

## 4. Gemma

Gemma is Google's open model. Two sizes: 2B and 7B.

**Architecture highlights vs LLaMA:**
- `scale_embeddings=True` — input embeddings multiplied by `sqrt(n_embd)` before the first layer
- Gemma 2B uses MQA (1 KV head); Gemma 7B uses a custom `head_size=256` (not `n_embd // n_head`)
- GemmaMLP uses GELU with tanh approximation instead of SiLU

**Auth required:** Set `HF_TOKEN` environment variable (free HuggingFace account + model access request).

The tokenizer (`GemmaTokenizer`) auto-downloads alongside the weights and registers
Gemma-specific special tokens including `<start_of_turn>` and `<end_of_turn>`.

In [ ]:
import os

# Set your HuggingFace token — needed to download Gemma weights
# os.environ['HF_TOKEN'] = 'hf_...'

if not os.environ.get('HF_TOKEN'):
    print('HF_TOKEN not set — skipping Gemma load.')
    print('Set it with: os.environ["HF_TOKEN"] = "hf_..."')
else:
    llm_gemma = LLM.from_pretrained(
        'gemma-2b',
        torch_dtype=torch.float16,
    )
    print(llm_gemma)

In [ ]:
if os.environ.get('HF_TOKEN'):
    gen_cfg = GenerationConfig(
        max_length=100,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        do_sample=True,
    )

    result = llm_gemma.generate_text(
        'Explain gradient descent in simple terms:',
        generation_config=gen_cfg,
        skip_prompt=True,
    )
    print(result['text'])

## 5. Inspect Gemma special tokens

`GemmaTokenizer` registers instruction-format tokens from the SentencePiece model.
On base Gemma these are present but unused; on Gemma-IT they mark turn boundaries.

In [ ]:
if os.environ.get('HF_TOKEN'):
    tok = llm_gemma.tokenizer.tokenizer  # unwrap TokenizerWrapper → GemmaTokenizer
    print('Gemma special tokens:')
    for name, token_id in tok.special_tokens.items():
        print(f'  {name!r:30s} → {token_id}')

## 6. Available models at a glance

In [ ]:
from myllm import LLM

llm = LLM()
print('All registered model variants:')
for family, variants in llm.list_models().items():
    for v in variants:
        print(f'  {v}')